# IC Textbook Dataset SFT数据处理

将原始数据集转换为SFT微调格式

## 1. 安装依赖

In [12]:
# 安装必要的依赖
!pip install -q openai datasets tqdm socksio httpx[socks] httpcore[socks]

## 2. 导入库

In [13]:
import os
# 禁用代理
os.environ['HTTP_PROXY'] = ''
os.environ['HTTPS_PROXY'] = ''
os.environ['http_proxy'] = ''
os.environ['https_proxy'] = ''
import json
import re
from typing import Dict, List, Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

from datasets import load_dataset
from openai import OpenAI
from tqdm import tqdm

## 3. 配置参数

In [ ]:
# ==================== 配置参数 ====================
import os

# OpenAI API 配置
API_KEY = os.getenv("OPENAI_API_KEY", "sk-MTIzLTExNzAwNDQwMDgzLTE3NzQ1ODMyMTEwOTc=")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.scnet.cn/api/llm/v1")  # 如果使用第三方API（如硅基流动），设置此项
MODEL = "MiniMax-M2.5"  # 或其他模型如 "gpt-3.5-turbo", "Qwen/Qwen2.5-72B-Instruct"

# 数据集路径
BASE_DATA_DIR = os.getenv("LLM_AGENT_DATA_DIR", "/data/llm_agent/data")
BASE_OUTPUT_DIR = os.getenv("LLM_AGENT_OUTPUT_DIR", "/data/llm_agent/output")
DATASET_NAME = "SeekBCD/IC-Textbook_Dataset"
CACHE_DIR = os.path.join(BASE_DATA_DIR, "IC_Textbook_Dataset")
OUTPUT_PATH = os.path.join(BASE_OUTPUT_DIR, "sft_data/ic_sft_data.jsonl")

# 处理参数
MAX_WORKERS = 5          # 并行线程数
MAX_ITEMS = 4000           # 处理条目数限制（设为None处理全部数据）
SAVE_INTERVAL = 100      # 每多少条保存一次
MIN_TEXT_LENGTH = 50     # 最小文本长度

print(f"API Key: {'已设置' if API_KEY != 'your-api-key-here' else '未设置'}")
print(f"Model: {MODEL}")
print(f"Max Items: {MAX_ITEMS if MAX_ITEMS else '全部'}")

## 4. 语言检测器

In [15]:
class LanguageDetector:
    """使用LLM的语言检测器"""

    # 语言代码映射
    LANG_MAP = {
        'Chinese': 'Chinese',
        'English': 'English',
        'Japanese': 'Japanese',
        'Korean': 'Korean',
        'French': 'French',
        'German': 'German',
        'Spanish': 'Spanish',
        'Russian': 'Russian',
        'Italian': 'Italian',
        'Portuguese': 'Portuguese',
    }

    def __init__(self, client: OpenAI, model: str):
        self.client = client
        self.model = model

    def detect(self, text: str) -> str:
        """使用LLM检测文本语言"""
        try:
            # 只取前500个字符进行检测，减少token消耗
            sample = text[:500] if len(text) > 500 else text

            prompt = f"""请判断以下文本的主要语言是什么。

文本内容：
{sample}

请只输出以下选项之一（不要有任何其他文字）：
Chinese, English, Japanese, Korean, French, German, Spanish, Russian, Italian, Portuguese

如果文本是中文（简体或繁体），输出：Chinese
如果文本是英文，输出：English
如果文本是日文，输出：Japanese
如果文本是韩文，输出：Korean
"""

            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a language detection assistant. Only respond with the language name in English."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.1,
                timeout=30  # 添加超时
            )

            # 安全地获取响应内容

            print(f"LLM response for language detection: {response}")  # 调试输出完整响应
            if response is None or not response.choices:
                print(f"Warning: Empty response from LLM for language detection (response={response})")
                return self._fallback_detect(text)

            message = response.choices[0].message
            if message is None:
                print("Warning: No message in LLM response for language detection")
                return self._fallback_detect(text)

            content = message.content
            if content is None:
                # 检查finish_reason
                finish_reason = response.choices[0].finish_reason
                print(f"Warning: No content in LLM response. Finish reason: {finish_reason}")
                return self._fallback_detect(text)

            lang_result = content.strip()

            # 清理可能的额外字符
            lang_result = lang_result.strip().rstrip('.').rstrip('。')

            # 映射到标准语言名称
            for std_name in self.LANG_MAP.keys():
                if std_name.lower() in lang_result.lower():
                    return std_name

            # 如果LLM返回的语言不在映射中，使用备用检测
            print(f"Warning: Unrecognized language '{lang_result}', using fallback")
            return self._fallback_detect(text)

        except Exception as e:
            print(f"Language detection error: {type(e).__name__}: {e}")
            return self._fallback_detect(text)

    def _fallback_detect(self, text: str) -> str:
        """备用语言检测（基于字符特征）"""
        # 检查中文
        if self._contains_chinese(text):
            return 'Chinese'
        # 检查日文
        if self._contains_japanese(text):
            return 'Japanese'
        # 检查韩文
        if self._contains_korean(text):
            return 'Korean'
        # 默认英文
        return 'English'

    @staticmethod
    def _contains_chinese(text: str) -> bool:
        """检查文本是否包含中文字符"""
        for char in text:
            if '\u4e00' <= char <= '\u9fff':
                return True
        return False

    @staticmethod
    def _contains_japanese(text: str) -> bool:
        """检查文本是否包含日文字符（平假名/片假名）"""
        for char in text:
            if '\u3040' <= char <= '\u309F' or '\u30A0' <= char <= '\u30FF':
                return True
        return False

    @staticmethod
    def _contains_korean(text: str) -> bool:
        """检查文本是否包含韩文字符"""
        for char in text:
            if '\uAC00' <= char <= '\uD7AF' or '\u1100' <= char <= '\u11FF':
                return True
        return False

## 5. Prompt模板

In [16]:
class PromptTemplates:
    """不同语言的Prompt模板"""

    @classmethod
    def get_prompt(cls, language: str, text: str) -> str:
        """根据语言获取对应的prompt"""
        prompts = {
            'Chinese': cls._chinese_prompt(text),
            'English': cls._english_prompt(text),
            'Japanese': cls._japanese_prompt(text),
            'Korean': cls._korean_prompt(text),
        }
        return prompts.get(language, cls._english_prompt(text))

    @staticmethod
    def _chinese_prompt(text: str) -> str:
        return f"""请基于以下文本内容，生成一个高质量的问答对用于模型微调。

                文本内容：
                {text[:3000]}

                请按以下要求生成：
                1. **question**: 基于文本内容生成一个具体、清晰的问题。问题应该能够测试对文本的理解深度。
                2. **cot** (思维链): 提供详细的思考过程，说明如何逐步分析文本并得出答案。
                3. **answer**: 基于文本内容提供完整、准确的回答。如果原文回答不够清晰或完整，请适当重写使其更适合作为SFT训练数据。
                4. **category**: 从以下类别中选择一个最符合的："基本概念", "技术原理", "应用场景", "设计方法", "制造工艺", "性能分析", "器件物理", "电路设计", "EDA工具", "其他"

                请以JSON格式输出，不要包含任何其他说明文字：
                {{
                    "question": "生成的问题",
                    "cot": "详细的思考过程...",
                    "answer": "回答内容...",
                    "category": "类别"
                }}"""

    @staticmethod
    def _english_prompt(text: str) -> str:
        return f"""Please generate a high-quality question-answer pair based on the following text for model fine-tuning.

                Text content:
                {text[:3000]}

                Please generate according to the following requirements:
                1. **question**: Generate a specific, clear question based on the text content. The question should test deep understanding of the text.
                2. **cot** (Chain of Thought): Provide a detailed reasoning process, explaining how to analyze the text step by step and arrive at the answer.
                3. **answer**: Provide a complete and accurate answer based on the text content. If the original answer is not clear or complete, please rewrite it appropriately to make it more suitable as SFT training data.
                4. **category**: Choose the most appropriate one from the following categories: "Basic Concepts", "Technical Principles", "Application Scenarios", "Design Methods", "Manufacturing Process", "Performance Analysis", "Device Physics", "Circuit Design", "EDA Tools", "Others"

                Please output in JSON format without any other explanatory text:
                {{
                    "question": "Generated question",
                    "cot": "Detailed reasoning process...",
                    "answer": "Answer content...",
                    "category": "Category"
                }}"""

    @staticmethod
    def _japanese_prompt(text: str) -> str:
        return f"""以下のテキスト内容に基づいて、モデルファインチューニング用の高品質なQ&Aペアを生成してください。

                テキスト内容：
                {text[:3000]}

                以下の要件に従って生成してください：
                1. **question**: テキスト内容に基づいて具体的で明確な質問を生成してください。
                2. **cot** (思考過程): テキストを段階的に分析して答えに至るまでの詳細な思考過程を提供してください。
                3. **answer**: テキスト内容に基づいて完全で正確な回答を提供してください。
                4. **category**: 以下のカテゴリーから最も適切なものを選んでください："基本概念", "技術原理", "応用シナリオ", "設計方法", "製造プロセス", "性能分析", "デバイス物理", "回路設計", "EDAツール", "その他"

                JSON形式で出力し、説明文は含めないでください：
                {{
                    "question": "生成された質問",
                    "cot": "詳細な思考過程...",
                    "answer": "回答内容...",
                    "category": "カテゴリー"
                }}"""

    @staticmethod
    def _korean_prompt(text: str) -> str:
        return f"""다음 텍스트 내용을 바탕으로 모델 미세조정을 위한 고품질 Q&A 쌍을 생성해주세요.

                텍스트 내용:
                {text[:3000]}

                다음 요구사항에 따라 생성해주세요:
                1. **question**: 텍스트 내용을 바탕으로 구체적이고 명확한 질문을 생성해주세요.
                2. **cot** (사고 과정): 텍스트를 단계적으로 분석하여 답에 이르는 자세한 사고 과정을 제공해주세요.
                3. **answer**: 텍스트 내용을 바탕으로 완전하고 정확한 답변을 제공해주세요.
                4. **category**: 다음 카테고리 중 가장 적절한 것을 선택해주세요: "기본 개념", "기술 원리", "응용 시나리오", "설계 방법", "제조 공정", "성능 분석", "소자 물리", "회로 설계", "EDA 도구", "기타"

                JSON 형식으로 출력하고 설명은 포함하지 마세요：
                {{
                    "question": "생성된 질문",
                    "cot": "자세한 사고 과정...",
                    "answer": "답변 내용...",
                    "category": "카테고리"
                }}"""

## 6. 数据处理器

In [17]:
class DataProcessor:
    """数据处理主类"""

    def __init__(self, api_key: str, base_url: Optional[str] = None, model: str = "gpt-4o"):
        self.client = OpenAI(api_key=api_key, base_url=base_url)
        self.model = model
        # 初始化语言检测器（使用LLM）
        self.language_detector = LanguageDetector(self.client, self.model)

    def process_single_item(
        self,
        text: str,
        source: Optional[str] = None,
        domain: Optional[str] = None
    ) -> Optional[Dict]:
        """处理单个文本条目"""
        try:
            # 1. 使用LLM检测语言
            language = self.language_detector.detect(text)

            # 2. 获取对应语言的prompt
            prompt = PromptTemplates.get_prompt(language, text)

            # 3. 调用LLM生成QA
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant specialized in generating high-quality training data for fine-tuning language models. Always respond with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=2000
            )

            # 4. 解析响应
            content = response.choices[0].message.content.strip()
            content = self._clean_json_response(content)
            qa_data = json.loads(content)

            # 5. 构建最终输出格式
            result = {
                "question": qa_data.get("question", ""),
                "cot": qa_data.get("cot", ""),
                "answer": qa_data.get("answer", ""),
                "language": language,
                "category": qa_data.get("category", "其他"),
                "source": source,
                "domain": domain,
                "original_text_preview": text[:500] if len(text) > 500 else text
            }

            # 验证必要字段
            if not result["question"] or not result["answer"]:
                print(f"Warning: Missing question or answer")
                return None

            return result

        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            print(f"Raw content: {content[:200] if 'content' in locals() else 'N/A'}")
            return None
        except Exception as e:
            print(f"Error processing item: {e}")
            return None

    @staticmethod
    def _clean_json_response(content: str) -> str:
        """清理LLM响应中的markdown代码块"""
        content = re.sub(r'^```json\s*', '', content)
        content = re.sub(r'^```\s*', '', content)
        content = re.sub(r'\s*```$', '', content)
        return content.strip()

    def process_dataset(
        self,
        dataset,
        output_path: str,
        max_workers: int = 5,
        max_items: Optional[int] = None,
        save_interval: int = 100,
        min_text_length: int = 50
    ) -> str:
        """批量处理数据集"""
        results = []
        processed_count = 0
        success_count = 0

        # 确定要处理的数据范围
        total_items = len(dataset) if max_items is None else min(max_items, len(dataset))
        items_to_process = dataset.select(range(total_items))

        print(f"开始处理 {total_items} 条数据...")
        print(f"输出文件: {output_path}")
        print(f"并行工作线程: {max_workers}")

        # 注意：语言检测现在使用LLM，每个item需要2次API调用
        # 1次用于语言检测，1次用于生成QA
        # 为了控制并发，我们串行处理语言检测，然后并行处理QA生成

        print("第一步：检测所有文本的语言...")
        languages = []
        items_with_lang = []

        for idx, item in enumerate(tqdm(items_to_process, desc="Language Detection")):
            text = item.get('text', '')
            source = item.get('source')
            domain = item.get('domain')

            if not text or len(text.strip()) < min_text_length:
                continue

            language = self.language_detector.detect(text)
            languages.append(language)
            items_with_lang.append((idx, text, source, domain, language))

        print(f"语言检测完成，共 {len(items_with_lang)} 条有效数据")
        print("语言分布:")
        from collections import Counter
        for lang, count in Counter(languages).most_common():
            print(f"  {lang}: {count}")

        # 第二步：并行生成QA
        print("\n第二步：并行生成QA对...")
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            future_to_item = {}
            for idx, text, source, domain, language in items_with_lang:
                future = executor.submit(
                    self._generate_qa, text, source, domain, language
                )
                future_to_item[future] = (idx, text, source, domain, language)

            # 处理完成的任务
            with tqdm(total=len(future_to_item), desc="Generating QA") as pbar:
                for future in as_completed(future_to_item):
                    idx, text, source, domain, language = future_to_item[future]
                    try:
                        result = future.result(timeout=120)
                        if result:
                            results.append(result)
                            success_count += 1

                            if len(results) % save_interval == 0:
                                self._append_to_jsonl(results, output_path)
                                results = []
                    except Exception as e:
                        print(f"Error in future {idx}: {e}")

                    processed_count += 1
                    pbar.update(1)

        # 保存剩余的结果
        if results:
            self._append_to_jsonl(results, output_path)

        print(f"\n处理完成!")
        print(f"总条目: {total_items}")
        print(f"成功处理: {success_count}")
        print(f"失败/跳过: {total_items - success_count}")
        print(f"输出文件: {output_path}")

        return output_path

    def _generate_qa(
        self,
        text: str,
        source: Optional[str],
        domain: Optional[str],
        language: str
    ) -> Optional[Dict]:
        """生成QA对（用于并行处理）"""
        try:
            # 获取对应语言的prompt
            prompt = PromptTemplates.get_prompt(language, text)

            # 调用LLM生成QA
            response = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {"role": "system", "content": "You are a helpful assistant specialized in generating high-quality training data for fine-tuning language models. Always respond with valid JSON."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.7,
                max_tokens=2000
            )

            # 解析响应
            content = response.choices[0].message.content.strip()
            content = self._clean_json_response(content)
            qa_data = json.loads(content)

            # 构建最终输出格式
            result = {
                "question": qa_data.get("question", ""),
                "cot": qa_data.get("cot", ""),
                "answer": qa_data.get("answer", ""),
                "language": language,
                "category": qa_data.get("category", "其他"),
                "source": source,
                "domain": domain,
                "original_text_preview": text[:500] if len(text) > 500 else text
            }

            # 验证必要字段
            if not result["question"] or not result["answer"]:
                print(f"Warning: Missing question or answer")
                return None

            return result

        except json.JSONDecodeError as e:
            print(f"JSON decode error: {e}")
            return None
        except Exception as e:
            print(f"Error generating QA: {e}")
            return None

    @staticmethod
    def _append_to_jsonl(data: List[Dict], filepath: str):
        """将数据追加到jsonl文件"""
        os.makedirs(os.path.dirname(filepath) if os.path.dirname(filepath) else '.', exist_ok=True)
        mode = 'a' if os.path.exists(filepath) else 'w'
        with open(filepath, mode, encoding='utf-8') as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + '\n')

## 7. 加载数据集

In [ ]:
# 加载IC Textbook数据集
import os

BASE_DATA_DIR = os.getenv("LLM_AGENT_DATA_DIR", "/data/llm_agent/data")

print("正在加载数据集...")
dataset = load_dataset(DATASET_NAME, cache_dir=os.path.join(BASE_DATA_DIR, "IC_Textbook_Dataset"), split="train")
print(f"数据集加载完成，共 {len(dataset)} 条数据")

# 查看样本数据
print("\n样本数据:")
print(json.dumps(dataset[0], indent=2, ensure_ascii=False)[:500] + "...")

## 8. 测试单条数据处理

In [19]:
# 初始化处理器
processor = DataProcessor(api_key=API_KEY, base_url=BASE_URL, model=MODEL)

# 测试处理第一条数据
test_item = dataset[0]
test_result = processor.process_single_item(
    text=test_item['text'],
    source=test_item.get('source'),
    domain=test_item.get('domain')
)

if test_result:
    print("处理结果:")
    print(json.dumps(test_result, indent=2, ensure_ascii=False))
else:
    print("处理失败")

LLM response for language detection: ChatCompletion(id='chatcmpl-c36ec68b0b8556ab219db2453a1fb64f', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nEnglish', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The user asks: "请判断以下文本的主要语言是什么。" The text is:\n\n"ains fundamental for fabricating masks used in optical projection systems (including EUVL), and NIL is important in "More than Moore" applications and in specific semiconductor devices like memories.\n\n**References**\n[Reference list remains unchanged from original]\n\n**Author Biography**\nDr. Alberto Roncaglia received an M.Sc. degree in Electrical Engineering and a Ph.D. degree in Electronics and Computer Science in 1998 and 2002, respectively, from the University of Bologna. In 2001, he joined"\n\nThis is clearly English. The user wants only output one of the options: Chinese, English, Japanese, Korean, French, Ge

## 9. 批量处理数据集

In [20]:
# 批量处理数据集
processor.process_dataset(
    dataset=dataset,
    output_path=OUTPUT_PATH,
    max_workers=MAX_WORKERS,
    max_items=MAX_ITEMS,  # 设为 None 处理全部数据
    save_interval=SAVE_INTERVAL,
    min_text_length=MIN_TEXT_LENGTH
)

开始处理 10 条数据...
输出文件: ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/sft_data/ic_sft_data.jsonl
并行工作线程: 5
第一步：检测所有文本的语言...


Language Detection:  10%|█         | 1/10 [00:03<00:27,  3.04s/it]

LLM response for language detection: ChatCompletion(id='chatcmpl-28eec5c3201468ce4e442ce372aa37b0', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nEnglish', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The user asks: "请判断以下文本的主要语言是什么。" The text is:\n\n"ains fundamental for fabricating masks used in optical projection systems (including EUVL), and NIL is important in "More than Moore" applications and in specific semiconductor devices like memories.\n\n**References**\n[Reference list remains unchanged from original]\n\n**Author Biography**\nDr. Alberto Roncaglia received an M.Sc. degree in Electrical Engineering and a Ph.D. degree in Electronics and Computer Science in 1998 and 2002, respectively, from the University of Bologna. In 2001, he joined"\n\nThis is clearly English. The user wants only one of the options: Chinese, English, Japanese, Korean, French, German, S

Language Detection:  20%|██        | 2/10 [00:06<00:26,  3.37s/it]

LLM response for language detection: ChatCompletion(id='chatcmpl-66d40ceab3cb1499855f247be28a2568', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nEnglish', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The user asks: "请判断以下文本的主要语言是什么。" The text is:\n\n"The general integer-N phase-locked loop (PLL) architecture is shown in Figure 19.11. This architecture produces an output frequency given by:\n\n\\[ f_{\\text{osc}} = \\frac{N}{M} f_{\\text{in}} \\]\n\nIn this case, the output frequency resolution requirement is 20 MHz. A divide-by-2 circuit (\\( M = 2 \\)) is applied to the 40 MHz input. To cover the full range of output frequencies, \\( N \\) must be programmable to any integer from 70 to 80. A value of \\( Q = 0.5 \\) is selected. Assuming the input referen"\n\nThe text is clearly English. The user wants only one of the listed language names: Chinese, English, Japan

...

Language Detection: 100%|██████████| 10/10 [01:07<00:00,  6.74s/it]


LLM response for language detection: ChatCompletion(id='chatcmpl-c196c49f6ae254f515fd6f83948bddb1', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\nEnglish', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[], reasoning='The user asks: "请判断以下文本的主要语言是什么。" The text is:\n\n"and Paolo Tessariol\n3. BCD Process Technologies – Giuseppe Croce, Antonio Andreini, Paola Galbiati, and Claudio Diazzi\n4. Measuring Techniques for Semiconductor Parameters – Alessandra Alberti et al.\n5. Interconnect Processing: Integration, Dielectrics, Metals – Shyng-Tsong Chen et al.\n6. Wet Chemical Processes for BEOL Technology – Cornelius Brown Peethala et al.\n7. From FinFET to Nanosheets and Beyond – Nadine Collaert\n8. Advanced Lithography – Alberto Roncaglia\n9. Advanced Technologies for Fu"\n\nThis is clearly English. The user wants only output one of the options: Chinese, English, Japanese, Korean, French

Generating QA: 100%|██████████| 10/10 [01:01<00:00,  6.11s/it]


处理完成!
总条目: 10
成功处理: 10
失败/跳过: 0
输出文件: ${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/sft_data/ic_sft_data.jsonl


'${LLM_AGENT_OUTPUT_DIR:-/data/llm_agent/output}/sft_data/ic_sft_data.jsonl'

## 10. 验证输出结果

In [21]:
# 读取并验证生成的文件
output_data = []
with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        output_data.append(json.loads(line))

print(f"共生成 {len(output_data)} 条SFT数据")
print("\n样本数据:")
print(json.dumps(output_data[0], indent=2, ensure_ascii=False))

共生成 30 条SFT数据

样本数据:
{
  "question": "What are the main challenges facing nanoelectronic devices and what solutions are proposed to address them according to the chapter on Advanced Technologies for Future Materials and Devices?",
  "cot": "To answer this question, I need to carefully analyze the text. First, I identify the challenges mentioned in the introduction section: increased power consumption, performance saturation, large variability, and reliability concerns. These are described as substantial challenges for the technology of future nanoelectronic devices. Next, I look for the proposed solutions in the abstract and keywords. The text mentions several promising solutions: multi-gate devices, nanowires, tunnel transistors, ferroelectric FETs, and hybrid nanocomponents using materials such as phase change materials or nanofilaments. Additionally, alternative materials including ultra-thin films, 2D and 1D nanostructures, and heterostructures using strained and III-V materials ar

## 11. 统计信息

In [22]:
from collections import Counter

# 统计语言分布
languages = [item['language'] for item in output_data]
print("语言分布:")
for lang, count in Counter(languages).most_common():
    print(f"  {lang}: {count}")

# 统计类别分布
categories = [item['category'] for item in output_data]
print("\n类别分布:")
for cat, count in Counter(categories).most_common():
    print(f"  {cat}: {count}")

语言分布:
  English: 26
  Chinese: 4

类别分布:
  Technical Principles: 19
  Device Physics: 3
  Basic Concepts: 2
  电路设计: 2
  基本概念: 2
  Circuit Design: 1
  Application Scenarios: 1
